Non Local Means
===============

*Important:* Please read the [installation page](http://gpeyre.github.io/numerical-tours/installation_python/) for details about how to install the toolboxes.
$\newcommand{\dotp}[2]{\langle #1, #2 \rangle}$
$\newcommand{\enscond}[2]{\lbrace #1, #2 \rbrace}$
$\newcommand{\pd}[2]{ \frac{ \partial #1}{\partial #2} }$
$\newcommand{\umin}[1]{\underset{#1}{\min}\;}$
$\newcommand{\umax}[1]{\underset{#1}{\max}\;}$
$\newcommand{\umin}[1]{\underset{#1}{\min}\;}$
$\newcommand{\uargmin}[1]{\underset{#1}{argmin}\;}$
$\newcommand{\norm}[1]{\|#1\|}$
$\newcommand{\abs}[1]{\left|#1\right|}$
$\newcommand{\choice}[1]{ \left\{  \begin{array}{l} #1 \end{array} \right. }$
$\newcommand{\pa}[1]{\left(#1\right)}$
$\newcommand{\diag}[1]{{diag}\left( #1 \right)}$
$\newcommand{\qandq}{\quad\text{and}\quad}$
$\newcommand{\qwhereq}{\quad\text{where}\quad}$
$\newcommand{\qifq}{ \quad \text{if} \quad }$
$\newcommand{\qarrq}{ \quad \Longrightarrow \quad }$
$\newcommand{\ZZ}{\mathbb{Z}}$
$\newcommand{\CC}{\mathbb{C}}$
$\newcommand{\RR}{\mathbb{R}}$
$\newcommand{\EE}{\mathbb{E}}$
$\newcommand{\Zz}{\mathcal{Z}}$
$\newcommand{\Ww}{\mathcal{W}}$
$\newcommand{\Vv}{\mathcal{V}}$
$\newcommand{\Nn}{\mathcal{N}}$
$\newcommand{\NN}{\mathcal{N}}$
$\newcommand{\Hh}{\mathcal{H}}$
$\newcommand{\Bb}{\mathcal{B}}$
$\newcommand{\Ee}{\mathcal{E}}$
$\newcommand{\Cc}{\mathcal{C}}$
$\newcommand{\Gg}{\mathcal{G}}$
$\newcommand{\Ss}{\mathcal{S}}$
$\newcommand{\Pp}{\mathcal{P}}$
$\newcommand{\Ff}{\mathcal{F}}$
$\newcommand{\Xx}{\mathcal{X}}$
$\newcommand{\Mm}{\mathcal{M}}$
$\newcommand{\Ii}{\mathcal{I}}$
$\newcommand{\Dd}{\mathcal{D}}$
$\newcommand{\Ll}{\mathcal{L}}$
$\newcommand{\Tt}{\mathcal{T}}$
$\newcommand{\si}{\sigma}$
$\newcommand{\al}{\alpha}$
$\newcommand{\la}{\lambda}$
$\newcommand{\ga}{\gamma}$
$\newcommand{\Ga}{\Gamma}$
$\newcommand{\La}{\Lambda}$
$\newcommand{\si}{\sigma}$
$\newcommand{\Si}{\Sigma}$
$\newcommand{\be}{\beta}$
$\newcommand{\de}{\delta}$
$\newcommand{\De}{\Delta}$
$\newcommand{\phi}{\varphi}$
$\newcommand{\th}{\theta}$
$\newcommand{\om}{\omega}$
$\newcommand{\Om}{\Omega}$

This numerical tour study image denoising using
non-local means. This algorithm has been
introduced for denoising purposes in [BuaCoMoA05](#biblio)

In [ ]:
from __future__ import division
import nt_toolbox as nt
from nt_solutions import denoisingadv_6_nl_means as solutions
%matplotlib inline
%load_ext autoreload
%autoreload 2

Patches in Images
-----------------
This numerical tour is dedicated to the study of the structure of patches
in images.


Size $N = n \times n$ of the image.

In [ ]:
n = 128

We load a noisy image $f_0\in \RR^N$.

In [ ]:
c = [100 200]
f0 = load_image('lena')
f0 = rescale(crop(f0, n, c))

Display $f_0$.

In [ ]:
imageplot(f0)

Noise level $\si$.

In [ ]:
sigma = .04

Generate a noisy image $f=f_0+\epsilon$ where $\epsilon \times
\Nn(0,\si^2\text{Id}_N)$.

In [ ]:
f = f0 + randn(n, n)*sigma

Display $f$.

In [ ]:
imageplot(clamp(f))

We denote $w$ to be the half width of the patches,
and $w_1=2w+1$ the full width.

In [ ]:
w = 3
w1 = 2*w + 1

We set up large $(n,n,w_1,w_1)$ matrices to index the the X and Y
position of the pixel to extract.

location of pixels

In [ ]:
[Y, X] = meshgrid(1: n, 1: n)

offsets

In [ ]:
[dY, dX] = meshgrid(-w: w, -w: w)

location of pixels to extract

In [ ]:
dX = reshape(dX, [1 1 w1 w1])
dY = reshape(dY, [1 1 w1 w1])
X = repmat(X, [1 1 w1 w1]) + repmat(dX, [n n 1 1])
Y = repmat(Y, [1 1 w1 w1]) + repmat(dY, [n n 1 1])

We handle boundary condition by reflexion

In [ ]:
X(X <1) = 2-X(X <1); Y(Y <1) = 2-Y(Y <1)
X(X >n) = 2*n-X(X >n); Y(Y >n) = 2*n-Y(Y >n)

Patch extractor operator

In [ ]:
patch = lambda f: f(X + (Y-1)*n)

Define the patch matrix $P$ of size $(n,n,w_1,w_1)$.
Each |P(i,j,:,:)| represent an $(w_1,w_1)$ patch extracted around pixel
$(i,j)$ in the image.

In [ ]:
P = patch(f)

Display some example of patches

In [ ]:
for i in 1: 16:
    x = floor(rand*(n-1) + 1)
    y = floor(rand*(n-1) + 1)
    imageplot(squeeze(P(x, y, : , : )), '', 4, 4, i)

Dimensionality Reduction with PCA
---------------------------------
Since NL-means type algorithms require the computation of many distances
between patches, it is advantagous to reduce the dimensionality of the
patch while keeping as much as possible of information.


Target dimensionality $d$.

In [ ]:
d = 25

A linear dimensionality reduction is obtained by Principal Component
Analysis (PCA) that projects the data on a small number of leading
direction of the coveriance matrix of the patches.


Turn the patch matrix into an |(w1*w1,n*n)| array, so that each |P(:,i)|
is a |w1*w1| vector representing a patch.

In [ ]:
resh = lambda P: reshape(P, [n*n w1*w1])'

operator to remove the mean of the patches to each patch.

In [ ]:
remove_mean = lambda Q: Q - repmat(mean(Q), [w1*w1 1])

Compute the mean and the covariance of the points cloud representing the
patches.

In [ ]:
P1 = remove_mean(resh(P))
C = P1*P1'

Extract the eigenvectors, sorted by decreasing amplitude

In [ ]:
[V, D] = eig(C); D = diag(D)
[D, I] = sort(D, 'descend'); V = V(: , I)

Display the decaying amplitude of the eigenvalues.

In [ ]:
plot(D); axis('tight')

Display the leading eigenvectors - they look like Fourier modes.

In [ ]:
for i in 1: 16:
    imageplot(reshape(V(: , i), [w1 w1]), '', 4, 4, i)

Patch dimensionality reduction operator.

In [ ]:
iresh = lambda Q: reshape(Q', [n n d])
descriptor = lambda f: iresh(V(: , 1: d)' * remove_mean(resh(P)))

Each |H(i,j,:)| is a $d$-dimensional descriptor
of a patch.

In [ ]:
H = descriptor(f)

Non-local Filter
----------------
NL-means applies, to each pixel location, an adaptive averaging kernel
that is computed from patch distances.


We denote $H_{i} \in \RR^d$ the descriptor at pixel $i$.
We define the distance matrix
$$ D_{i,j} = \frac{1}{w_1^2}\norm{H_i-H_j}^2. $$



Operator to compute the distances $(D_{i,j})_j$ between the patch around $i=(i_1,i_2)$
and all the other ones.

In [ ]:
distance = lambda i: sum((H - repmat(H(i(1), i(2), : ), [n n 1])).^2, 3)/ (w1*w1)

The non-local mean filter compute a denoised image $\tilde f$ as
$$ \tilde f_i = \sum_j K_{i,j} f_j $$
where the weights $K$ are computed as
$$ K_{i,j} = \frac{ \tilde K_{i,j} }{ \sum_{j'} \tilde K_{i,j'} }
      \qandq
   \tilde K_{i,j} = e^{-\frac{D_{i,j}}{2\tau^2}} . $$



The width $\tau$ of the Gaussian is very important and should be adapted to match
the noise level.



Compute and normalize the weight.

In [ ]:
normalize = lambda K: K/ sum(K(: ))
kernel = lambda i, tau: normalize(exp(-distance(i)/ (2*tau^2)))

Compute a typical example of kernel for some pixel position $(x,y)$.

In [ ]:
tau = .05
i = [83 72]
D = distance(i)
K = kernel(i, tau)

Display the squared distance and the kernel.

In [ ]:
imageplot(D, 'D', 1, 2, 1)
imageplot(K, 'K', 1, 2, 2)

Localizing the Non-local Means
------------------------------
We set a "locality constant" $q$ that set the maximum distance between
patches to compare. This allows to speed up computation, and makes
NL-means type methods semi-global (to avoid searching in all the image).

In [ ]:
q = 14

Using this locality constant, we compute the distance between patches
only within a window.
Once again, one should be careful about boundary conditions.

In [ ]:
selection = lambda i: {clamp(i(1)-q: i(1) + q, 1, n), clamp(i(2)-q: i(2) + q, 1, n)}

Compute distance and kernel only within the window.

In [ ]:
distance = lambda i, sel: sum((H(sel{1}, sel{2}, : ) - repmat(H(i(1), i(2), : ), ...
        [length(sel{1}) length(sel{2}) 1])).^2, 3)/ (w1*w1)
distance = lambda i: distance(i, selection(i))
kernel = lambda i, tau: normalize(exp(-distance(i)/ (2*tau^2)))

Compute a typical example of kernel for some pixel position $(x,y)$.

In [ ]:
D = distance(i)
K = kernel(i, tau)

Display the squared distance and the kernel.

In [ ]:
imageplot(D, 'D', 1, 2, 1)
imageplot(K, 'K', 1, 2, 2)

The NL-filtered value at pixel $(x,y)$ is obtained by averaging the values
of $f$ with the weight $K$.

In [ ]:
NLval = lambda K, sel: sum(sum(K.*f(sel{1}, sel{2})))
NLval = lambda i, tau: NLval(kernel(i, tau), selection(i))

We apply the filter to each pixel location
to perform the NL-means algorithm.

In [ ]:
[Y, X] = meshgrid(1: n, 1: n)
NLmeans = lambda tau: arrayfun(lambda i1, i2: NLval([i1 i2], tau), X, Y)

Display the result for some value of $\tau$.

In [ ]:
tau = .03

imageplot(NLmeans(tau))

__Exercise 1__

Compute the denoising result for several values of $\tau$ in order to
determine the optimal denoising that minimizes $\norm{\tilde f - f_0}$.

In [ ]:
solutions.exo1()

In [ ]:
## Insert your code here.

Display the best result.

In [ ]:
imageplot(clamp(fNL))

__Exercise 2__

Explore the influence of the $q$ and $w$ parameters.

In [ ]:
solutions.exo2()

In [ ]:
## Insert your code here.

Bibliography
------------
<html><a name="biblio"></a></html>


* [BuaCoMoA05] Buades, B. Coll, J.f Morel, [A review of image denoising algorithms, with a new one][1], SIAM Multiscale Modeling and Simulation, Vol 4 (2), pp: 490-530, 2005.

[1]:http://dx.doi.org/10.1137/040616024